Paths and reusable definitions were refactored. Stored outputs below are historical, not a fresh execution. See `docs/WORKFLOW.md` at the repository root before rerunning; several stages overwrite results or append Optuna trials.


In [ ]:
# Locate this checkout from the repository root or any notebook subdirectory.
from pathlib import Path
import sys

_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
              if (p / "src" / "steam_recommender" / "paths.py").is_file()), None)
if _root is None:
    raise RuntimeError("Start Jupyter inside the repository checkout.")
sys.path.insert(0, str(_root / "src"))
from steam_recommender import paths as _paths
if Path(_paths.__file__).resolve() != (_root / "src/steam_recommender/paths.py").resolve():
    raise RuntimeError("A different checkout is cached; restart the kernel.")
from steam_recommender.paths import project_root, project_path


In [1]:
import os

os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix

from implicit.als import AlternatingLeastSquares

In [3]:
# Notebook:
#
# ROOT_DIR:

ROOT_DIR = project_root()

RAW_DATA_DIR = project_path("data/raw")

REPLICATION_DIR = project_path("results/literature")
RESULTS_DIR = REPLICATION_DIR / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

USERS_ITEMS_PATH = RAW_DATA_DIR / "users_items.csv"
STEAM_GAMES_PATH = RAW_DATA_DIR / "steam_games_final.csv"

print("ROOT_DIR:", ROOT_DIR)
print("USERS_ITEMS_PATH:", USERS_ITEMS_PATH)
print("STEAM_GAMES_PATH:", STEAM_GAMES_PATH)

print("users_items exists:", USERS_ITEMS_PATH.exists())
print("steam_games exists:", STEAM_GAMES_PATH.exists())

ROOT_DIR: D:\Studiowe\Licencjat\modele
USERS_ITEMS_PATH: D:\Studiowe\Licencjat\modele\data_raw\users_items.csv
STEAM_GAMES_PATH: D:\Studiowe\Licencjat\modele\data_raw\steam_games_final.csv
users_items exists: True
steam_games exists: True


In [4]:
# Replikacja Bayram-style ALS, paid games only.

# Transformacja czasu:
# U nas playtime_forever jest w minutach.
# W pracy Bayram czas jest interpretowany w godzinach.
# Dlatego:
# playtime_hours = playtime_forever / 60
# signal = log(playtime_hours + 1) + 0.01

PLAYTIME_ADD_CONSTANT = 0.01

# Parametry modelu ALS z wariantu paid games.
# Najpierw ustawiamy model bliski ich najlepszemu ALS.
ALS_FACTORS = 100
ALS_REGULARIZATION = 1.0
ALS_ITERATIONS = 20
ALS_ALPHA = 6.0

K = 10
RANDOM_SEED = 0

print("ALS_FACTORS:", ALS_FACTORS)
print("ALS_REGULARIZATION:", ALS_REGULARIZATION)
print("ALS_ITERATIONS:", ALS_ITERATIONS)
print("ALS_ALPHA:", ALS_ALPHA)
print("K:", K)

ALS_FACTORS: 100
ALS_REGULARIZATION: 1.0
ALS_ITERATIONS: 20
ALS_ALPHA: 6.0
K: 10


In [5]:
users_items_raw = pd.read_csv(USERS_ITEMS_PATH)
steam_games = pd.read_csv(STEAM_GAMES_PATH)

print("users_items_raw shape:", users_items_raw.shape)
print("steam_games shape:", steam_games.shape)

display(users_items_raw.head())
display(steam_games.head())

users_items_raw shape: (5153209, 4)
steam_games shape: (27364, 6)


,user_id,item_id,item_name,playtime_forever
0,76561197970982479,10,Counter-Strike,6
1,76561197970982479,20,Team Fortress Classic,0
2,76561197970982479,30,Day of Defeat,7
3,76561197970982479,40,Deathmatch Classic,0
4,76561197970982479,50,Half-Life: Opposing Force,0


,genres,app_name,tags,price,id,About the game
0,"['Action', 'Casual', 'Indie', 'Simulation', 'S...",Lost Summoner Kitty,"['Strategy', 'Action', 'Indie', 'Casual', 'Sim...",4.99,761140,Monsters appearing one after another. Let's su...
1,"['Free to Play', 'Indie', 'RPG', 'Strategy']",Ironbound,"['Free to Play', 'Strategy', 'Indie', 'RPG', '...",Free To Play,643980,Sharpen your sword and join the fray in IRONBO...
2,"['Casual', 'Free to Play', 'Indie', 'Simulatio...",Real Pool 3D - Poolians,"['Free to Play', 'Simulation', 'Sports', 'Casu...",Free to Play,670290,Real Pool 3D (Poolians) is one of the best onl...
3,"['Action', 'Adventure', 'Casual']",弹炸人2222,"['Action', 'Adventure', 'Casual']",0.99,767400,"The author of the first game, a bit special， T..."
4,"['Action', 'Adventure', 'Simulation']",Battle Royale Trainer,"['Action', 'Adventure', 'Simulation', 'FPS', '...",3.99,772540,NaN


In [6]:
print("users_items columns:")
print(users_items_raw.columns.tolist())

print("\nsteam_games columns:")
print(steam_games.columns.tolist())

users_items columns:
['user_id', 'item_id', 'item_name', 'playtime_forever']

steam_games columns:
['genres', 'app_name', 'tags', 'price', 'id', 'About the game']


In [7]:
required_users_items_cols = [
    "user_id",
    "item_id",
    "item_name",
    "playtime_forever",
]

missing_cols = [
    col for col in required_users_items_cols
    if col not in users_items_raw.columns
]

if missing_cols:
    raise ValueError(f"Missing columns in users_items_raw: {missing_cols}")

if "id" not in steam_games.columns:
    raise ValueError("steam_games_final.csv musi mieć kolumnę 'id' odpowiadającą item_id.")

print("OK")

OK


In [8]:
steam_games_info = pd.DataFrame({
    "column": steam_games.columns,
    "dtype": [steam_games[col].dtype for col in steam_games.columns],
    "missing_count": [steam_games[col].isna().sum() for col in steam_games.columns],
    "missing_share": [steam_games[col].isna().mean() for col in steam_games.columns],
})

display(steam_games_info)

,column,dtype,missing_count,missing_share
0,genres,str,0,0.000000
1,app_name,str,0,0.000000
2,tags,str,0,0.000000
3,price,str,1143,0.041770
4,id,int64,0,0.000000
5,About the game,str,14732,0.538372


In [9]:
# Diagnostyka kolumn związanych z ceną / darmowością.
# Dodatkowo tworzymy price_numeric_clean:
# - liczby i stringi typu "9.99" zostają sparsowane jako float
# - missing values -> 0
# - wartości nienumeryczne -> 0

steam_games_price_diag = steam_games.copy()

candidate_price_cols = [
    col for col in steam_games_price_diag.columns
    if any(
        token in col.lower()
        for token in ["price", "free", "cost", "discount"]
    )
]

print("Candidate price/free columns:")
print(candidate_price_cols)

for col in candidate_price_cols:
    print("\n", "=" * 80)
    print(col)
    print("dtype:", steam_games_price_diag[col].dtype)
    print("nunique:", steam_games_price_diag[col].nunique(dropna=True))
    display(steam_games_price_diag[col].value_counts(dropna=False).head(20))


# Najpierw wybieramy kolumnę ceny.
# Preferujemy dokładnie 'price', potem dowolną kolumnę z 'price' w nazwie.

if "price" in steam_games_price_diag.columns:
    PRICE_COL = "price"
else:
    price_cols = [
        col for col in steam_games_price_diag.columns
        if "price" in col.lower()
    ]
    
    if not price_cols:
        raise ValueError(
            "Nie znaleziono kolumny ceny. "
            f"Kandydaci price/free/cost/discount: {candidate_price_cols}"
        )
    
    PRICE_COL = price_cols[0]

print("\nUsing PRICE_COL:", PRICE_COL)


# Parsowanie:
# - missing -> 0
# - liczby typu 9.99 albo stringi '9.99' -> float
# - wszystko inne -> NaN -> 0

price_as_str = (
    steam_games_price_diag[PRICE_COL]
    .fillna("0")
    .astype(str)
    .str.strip()
    .str.replace(",", ".", regex=False)
)

price_numeric_clean = pd.to_numeric(
    price_as_str,
    errors="coerce",
).fillna(0.0)

steam_games_price_diag["price_numeric_clean"] = price_numeric_clean

display(
    steam_games_price_diag[
        [PRICE_COL, "price_numeric_clean"]
    ].head(30)
)

print("\nprice_numeric_clean describe:")
display(steam_games_price_diag["price_numeric_clean"].describe())

print("\nTop raw price values:")
display(
    steam_games_price_diag[PRICE_COL]
    .value_counts(dropna=False)
    .head(30)
)

print("\nTop parsed price values:")
display(
    steam_games_price_diag["price_numeric_clean"]
    .value_counts(dropna=False)
    .head(30)
)

Candidate price/free columns:
['price']

price
dtype: str
nunique: 146


price
4.99            3690
9.99            3339
2.99            3123
0.99            2429
1.99            2194
19.99           1450
3.99            1369
14.99           1234
NaN             1143
6.99            1066
7.99             954
5.99             827
Free             658
29.99            459
Free to Play     419
12.99            341
24.99            339
Free To Play     325
11.99            280
8.99             277
Name: count, dtype: int64


Using PRICE_COL: price


,price,price_numeric_clean
0,4.99,4.99
1,Free To Play,0.00
2,Free to Play,0.00
3,0.99,0.99
4,3.99,3.99
5,9.99,9.99
6,18.99,18.99
7,29.99,29.99
8,NaN,0.00
9,10.99,10.99



price_numeric_clean describe:


count    27364.000000
mean         8.575891
std         15.974832
min          0.000000
25%          1.990000
50%          4.990000
75%          9.990000
max        995.000000
Name: price_numeric_clean, dtype: float64


Top raw price values:


price
4.99            3690
9.99            3339
2.99            3123
0.99            2429
1.99            2194
19.99           1450
3.99            1369
14.99           1234
NaN             1143
6.99            1066
7.99             954
5.99             827
Free             658
29.99            459
Free to Play     419
12.99            341
24.99            339
Free To Play     325
11.99            280
8.99             277
39.99            267
49.99            153
34.99            101
59.99             93
17.99             74
16.99             73
13.99             69
10.99             64
99.99             59
15.99             56
Name: count, dtype: int64


Top parsed price values:


price_numeric_clean
4.99     3690
9.99     3339
2.99     3123
0.00     2565
0.99     2429
1.99     2194
19.99    1450
3.99     1369
14.99    1234
6.99     1066
7.99      954
5.99      827
29.99     459
12.99     341
24.99     339
11.99     280
8.99      277
39.99     267
49.99     153
34.99     101
59.99      93
17.99      74
16.99      73
13.99      69
10.99      64
99.99      59
15.99      56
18.99      47
2.49       38
44.99      32
Name: count, dtype: int64

In [10]:
steam_games_paid = steam_games.copy()
steam_games_paid["id"] = steam_games_paid["id"].astype(str)

# Korzystamy z price_numeric_clean utworzonego w poprawionym bloku.
# Zasada:
# price_numeric_clean > 0 -> paid game
# price_numeric_clean == 0 -> free / missing / nienumeryczna cena

steam_games_paid["price_numeric_clean"] = steam_games_price_diag["price_numeric_clean"].values
steam_games_paid["is_paid"] = steam_games_paid["price_numeric_clean"] > 0

display(
    steam_games_paid["is_paid"]
    .value_counts(dropna=False)
    .rename_axis("is_paid")
    .reset_index(name="count")
)

display(
    steam_games_paid[
        ["id", "price_numeric_clean", "is_paid"]
    ].head(20)
)

,is_paid,count
0,True,24799
1,False,2565


,id,price_numeric_clean,is_paid
0,761140,4.99,True
1,643980,0.00,False
2,670290,0.00,False
3,767400,0.99,True
4,772540,3.99,True
5,774276,9.99,True
6,774277,18.99,True
7,774278,29.99,True
8,768800,0.00,False
9,770380,10.99,True


In [11]:
users_items = users_items_raw.copy()

users_items["item_id"] = users_items["item_id"].astype(str)
users_items["user_id"] = users_items["user_id"].astype(str)

paid_item_ids = set(
    steam_games_paid.loc[
        steam_games_paid["is_paid"],
        "id"
    ].astype(str)
)

users_items_paid = users_items[
    users_items["item_id"].isin(paid_item_ids)
].copy()

print("Raw users_items:", users_items.shape)
print("Paid users_items:", users_items_paid.shape)

print("Raw users:", users_items["user_id"].nunique())
print("Paid users:", users_items_paid["user_id"].nunique())

print("Raw items:", users_items["item_id"].nunique())
print("Paid items:", users_items_paid["item_id"].nunique())

print("Share interactions kept:", len(users_items_paid) / len(users_items))

Raw users_items: (5153209, 4)
Paid users_items: (3257824, 4)
Raw users: 70912
Paid users: 65855
Raw items: 10978
Paid items: 7550
Share interactions kept: 0.6321932605489123


In [12]:
def dataset_stats(df, name):
    n_interactions = len(df)
    n_users = df["user_id"].nunique()
    n_items = df["item_id"].nunique()
    density = n_interactions / (n_users * n_items)

    return {
        "stage": name,
        "interactions": n_interactions,
        "users": n_users,
        "items": n_items,
        "density": density,
    }


paid_stats = pd.DataFrame([
    dataset_stats(users_items, "raw_users_items"),
    dataset_stats(users_items_paid, "paid_games_only"),
])

display(paid_stats)

paid_stats.to_csv(
    RESULTS_DIR / "bayram2021_paid_dataset_stats.csv",
    index=False,
)

,stage,interactions,users,items,density
0,raw_users_items,5153209,70912,10978,0.006620
1,paid_games_only,3257824,65855,7550,0.006552


In [13]:
model_df = users_items_paid.copy()

duplicate_pairs = model_df.duplicated(
    subset=["user_id", "item_id"]
).sum()

print("Duplicate user-item pairs:", duplicate_pairs)
print("Duplicate share:", duplicate_pairs / len(model_df))

model_df = (
    model_df
    .groupby(["user_id", "item_id"], as_index=False)
    .agg({
        "item_name": "first",
        "playtime_forever": "sum",
    })
)

model_df["bayram_signal_minutes"] = (
    np.log(model_df["playtime_forever"] + 1.0)
    + PLAYTIME_ADD_CONSTANT
)

model_df["confidence"] = (
    ALS_ALPHA * model_df["bayram_signal_minutes"]
)

print("After aggregation:", model_df.shape)
print("Users:", model_df["user_id"].nunique())
print("Items:", model_df["item_id"].nunique())

display(model_df.head())

print("confidence describe:")
display(model_df["confidence"].describe())

Duplicate user-item pairs: 37038
Duplicate share: 0.011368938285186677
After aggregation: (3220786, 6)
Users: 65855
Items: 7550


,user_id,item_id,item_name,playtime_forever,bayram_signal_minutes,confidence
0,--000--,104900,ORION: Prelude,63,4.168883,25.013299
1,--000--,105600,Terraria,1172,7.077320,42.463919
2,--000--,1250,Killing Floor,2949,7.999560,47.997363
3,--000--,204360,Castle Crashers,1597,7.386508,44.319049
4,--000--,209270,Hero Academy,38,3.673562,22.041370


confidence describe:


count    3.220786e+06
mean     2.146940e+01
std      1.837740e+01
min      6.000000e-02
25%      6.000000e-02
50%      2.421211e+01
75%      3.649827e+01
max      8.100693e+01
Name: confidence, dtype: float64

In [14]:
user_interaction_counts = model_df["user_id"].value_counts()

valid_users = user_interaction_counts[
    user_interaction_counts >= 2
].index

model_df = model_df[
    model_df["user_id"].isin(valid_users)
].copy()

print("After min 2 interactions per user:")
print("Interactions:", len(model_df))
print("Users:", model_df["user_id"].nunique())
print("Items:", model_df["item_id"].nunique())

After min 2 interactions per user:
Interactions: 3216497
Users: 61566
Items: 7550


In [15]:
user_ids = model_df["user_id"].drop_duplicates().reset_index(drop=True)
item_ids = model_df["item_id"].drop_duplicates().reset_index(drop=True)

user_id_to_idx = pd.Series(
    data=np.arange(len(user_ids)),
    index=user_ids,
)

item_id_to_idx = pd.Series(
    data=np.arange(len(item_ids)),
    index=item_ids,
)

idx_to_user_id = pd.Series(
    data=user_ids.values,
    index=np.arange(len(user_ids)),
)

idx_to_item_id = pd.Series(
    data=item_ids.values,
    index=np.arange(len(item_ids)),
)

model_df["user_idx"] = model_df["user_id"].map(user_id_to_idx)
model_df["item_idx"] = model_df["item_id"].map(item_id_to_idx)

n_users = len(user_ids)
n_items = len(item_ids)

print("n_users:", n_users)
print("n_items:", n_items)

display(model_df.head())

n_users: 61566
n_items: 7550


,user_id,item_id,item_name,playtime_forever,bayram_signal_minutes,confidence,user_idx,item_idx
0,--000--,104900,ORION: Prelude,63,4.168883,25.013299,0,0
1,--000--,105600,Terraria,1172,7.077320,42.463919,0,1
2,--000--,1250,Killing Floor,2949,7.999560,47.997363,0,2
3,--000--,204360,Castle Crashers,1597,7.386508,44.319049,0,3
4,--000--,209270,Hero Academy,38,3.673562,22.041370,0,4


In [16]:
def one_hidden_item_split(df, seed=42):
    rng = np.random.default_rng(seed)

    df = df.copy()
    df["is_test"] = False

    for user_idx, user_part in df.groupby("user_idx"):
        indices = user_part.index.to_numpy().copy()
        test_index = rng.choice(indices)
        df.loc[test_index, "is_test"] = True

    train_df = df[~df["is_test"]].copy()
    test_df = df[df["is_test"]].copy()

    return train_df, test_df


train_df, test_df = one_hidden_item_split(
    model_df,
    seed=RANDOM_SEED,
)

print("Train interactions:", len(train_df))
print("Test interactions:", len(test_df))
print("Test users:", test_df["user_idx"].nunique())
print("Users total:", model_df["user_idx"].nunique())

display(train_df.head())
display(test_df.head())

Train interactions: 3154931
Test interactions: 61566
Test users: 61566
Users total: 61566


,user_id,item_id,item_name,playtime_forever,bayram_signal_minutes,confidence,user_idx,item_idx,is_test
0,--000--,104900,ORION: Prelude,63,4.168883,25.013299,0,0,False
1,--000--,105600,Terraria,1172,7.077320,42.463919,0,1,False
2,--000--,1250,Killing Floor,2949,7.999560,47.997363,0,2,False
3,--000--,204360,Castle Crashers,1597,7.386508,44.319049,0,3,False
4,--000--,209270,Hero Academy,38,3.673562,22.041370,0,4,False


,user_id,item_id,item_name,playtime_forever,bayram_signal_minutes,confidence,user_idx,item_idx,is_test
15,--000--,55230,Saints Row: The Third,1922,7.571642,45.429850,0,15,True
29,--ace--,343360,Particula,0,0.010000,0.060000,1,27,True
39,--ionex--,427730,Who's Your Daddy,19,3.005732,18.034394,2,34,True
55,-2SV-vuLB-Kg,274900,Murder Miners,0,0.010000,0.060000,3,47,True
108,-404PageNotFound-,224580,Arma 2: DayZ Mod,5,1.801759,10.810557,4,94,True


In [17]:
def build_user_item_matrix(df, n_users, n_items, value_col="confidence"):
    rows = df["user_idx"].to_numpy()
    cols = df["item_idx"].to_numpy()
    data = df[value_col].to_numpy(dtype=np.float32)

    matrix = csr_matrix(
        (data, (rows, cols)),
        shape=(n_users, n_items),
    )

    matrix.sum_duplicates()

    return matrix


user_item_train = build_user_item_matrix(
    train_df,
    n_users=n_users,
    n_items=n_items,
    value_col="confidence",
)

print(user_item_train.shape)
print("nnz:", user_item_train.nnz)
print("data min:", user_item_train.data.min())
print("data max:", user_item_train.data.max())
print("data mean:", user_item_train.data.mean())

(61566, 7550)
nnz: 3154931
data min: 0.06
data max: 81.00693
data mean: 21.329752


In [18]:
test_items_by_user = (
    test_df
    .groupby("user_idx")["item_idx"]
    .first()
    .to_dict()
)

print("Number of test users:", len(test_items_by_user))

train_counts_by_user = np.diff(user_item_train.indptr)

test_users_without_train = [
    user_idx for user_idx in test_items_by_user
    if train_counts_by_user[user_idx] == 0
]

print("Test users without train interactions:", len(test_users_without_train))

Number of test users: 61566
Test users without train interactions: 0


In [19]:
model = AlternatingLeastSquares(
    factors=ALS_FACTORS,
    regularization=ALS_REGULARIZATION,
    iterations=ALS_ITERATIONS,
    random_state=RANDOM_SEED,
    num_threads=8,
)

model.fit(user_item_train)

  0%|          | 0/20 [00:00<?, ?it/s]

In [20]:
def evaluate_one_hidden_item_hit_rate(
    model,
    user_item_train,
    test_items_by_user,
    k=10,
):
    hits = []
    evaluated_users = []

    for user_idx, hidden_item_idx in test_items_by_user.items():
        recommended_items, scores = model.recommend(
            userid=user_idx,
            user_items=user_item_train[user_idx],
            N=k,
            filter_already_liked_items=True,
        )

        hit = int(hidden_item_idx in set(recommended_items))

        hits.append(hit)
        evaluated_users.append(user_idx)

    results_df = pd.DataFrame({
        "user_idx": evaluated_users,
        "hidden_item_idx": [
            test_items_by_user[user_idx]
            for user_idx in evaluated_users
        ],
        "hit": hits,
    })

    hit_rate_at_k = np.mean(hits)

    return hit_rate_at_k, results_df

In [21]:
hit_rate_at_10, bayram_eval_results = evaluate_one_hidden_item_hit_rate(
    model=model,
    user_item_train=user_item_train,
    test_items_by_user=test_items_by_user,
    k=K,
)

print(f"Bayram-style Precision@{K} / HitRate@{K}: {hit_rate_at_10:.4f}")
print(f"Percentage: {hit_rate_at_10 * 100:.2f}%")

display(bayram_eval_results.head())
display(bayram_eval_results["hit"].value_counts(normalize=True))

Bayram-style Precision@10 / HitRate@10: 0.2918
Percentage: 29.18%


,user_idx,hidden_item_idx,hit
0,0,15,1
1,1,27,0
2,2,34,0
3,3,47,0
4,4,94,1


hit
0    0.708167
1    0.291833
Name: proportion, dtype: float64

In [22]:
def ndcg_one_hidden_item(recommended_items, hidden_item_idx, k=10):
    recommended_items = list(recommended_items[:k])

    if hidden_item_idx not in recommended_items:
        return 0.0

    rank = recommended_items.index(hidden_item_idx) + 1
    return 1.0 / np.log2(rank + 1)


def average_pairwise_cosine_distance(item_indices, item_factors):
    item_indices = list(item_indices)

    if len(item_indices) < 2:
        return np.nan

    vectors = item_factors[item_indices].astype(np.float64)

    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1.0

    vectors = vectors / norms

    similarities = vectors @ vectors.T

    upper_triangle_indices = np.triu_indices_from(
        similarities,
        k=1,
    )

    pairwise_similarities = similarities[upper_triangle_indices]
    pairwise_distances = 1.0 - pairwise_similarities

    return np.mean(pairwise_distances)

In [23]:
def evaluate_extended_one_hidden_item(
    model,
    user_item_train,
    train_df,
    test_items_by_user,
    n_items,
    k=10,
):
    train_item_counts = train_df["item_idx"].value_counts()
    train_item_distribution = train_item_counts / train_item_counts.sum()

    recommended_all = []

    hit_values = []
    precision_values = []
    recall_values = []
    ndcg_values = []
    diversity_values = []

    recommended_popularity_values = []

    per_user_rows = []

    for user_idx, hidden_item_idx in test_items_by_user.items():
        recommended_items, scores = model.recommend(
            userid=user_idx,
            user_items=user_item_train[user_idx],
            N=k,
            filter_already_liked_items=True,
        )

        recommended_items = list(recommended_items)

        hit = int(hidden_item_idx in recommended_items)
        precision = hit / k
        recall = hit
        ndcg = ndcg_one_hidden_item(
            recommended_items,
            hidden_item_idx,
            k=k,
        )

        diversity = average_pairwise_cosine_distance(
            recommended_items,
            model.item_factors,
        )

        recommended_all.extend(recommended_items)

        item_popularities = [
            train_item_distribution.get(item_idx, 0.0)
            for item_idx in recommended_items
        ]

        recommended_popularity_values.extend(item_popularities)

        hit_values.append(hit)
        precision_values.append(precision)
        recall_values.append(recall)
        ndcg_values.append(ndcg)
        diversity_values.append(diversity)

        per_user_rows.append({
            "user_idx": user_idx,
            "hidden_item_idx": hidden_item_idx,
            "hit": hit,
            f"precision@{k}": precision,
            f"recall@{k}": recall,
            f"ndcg@{k}": ndcg,
            f"diversity@{k}": diversity,
        })

    recommended_unique_items = len(set(recommended_all))
    coverage_at_k = recommended_unique_items / n_items

    gap_recommendations = np.mean(recommended_popularity_values)

    all_item_popularities = np.zeros(n_items)

    for item_idx, popularity in train_item_distribution.items():
        all_item_popularities[item_idx] = popularity

    gap_catalog = np.mean(all_item_popularities)
    delta_gap_at_k = gap_recommendations - gap_catalog

    summary = {
        "model": "Bayram-style ALS paid minutes",
        "users_evaluated": len(test_items_by_user),
        f"hit_rate@{k}": np.mean(hit_values),
        f"bayram_precision@{k}": np.mean(hit_values),
        f"precision@{k}": np.mean(precision_values),
        f"recall@{k}": np.mean(recall_values),
        f"ndcg@{k}": np.mean(ndcg_values),
        f"coverage@{k}": coverage_at_k,
        f"delta_gap@{k}": delta_gap_at_k,
        f"diversity@{k}": np.nanmean(diversity_values),
        "recommended_unique_items": recommended_unique_items,
        "n_items": n_items,
    }

    per_user_df = pd.DataFrame(per_user_rows)

    return summary, per_user_df

In [24]:
bayram_extended_summary, bayram_extended_per_user = evaluate_extended_one_hidden_item(
    model=model,
    user_item_train=user_item_train,
    train_df=train_df,
    test_items_by_user=test_items_by_user,
    n_items=n_items,
    k=K,
)

bayram_extended_summary_df = pd.DataFrame([bayram_extended_summary])

display(bayram_extended_summary_df)
display(bayram_extended_per_user.head())

bayram_extended_summary_df.to_csv(
    RESULTS_DIR / "bayram2021_paid_als_minutes_extended_summary.csv",
    index=False,
)

bayram_extended_per_user.to_csv(
    RESULTS_DIR / "bayram2021_paid_als_minutes_extended_per_user_metrics.csv",
    index=False,
)

,model,users_evaluated,hit_rate@10,bayram_precision@10,precision@10,recall@10,ndcg@10,coverage@10,delta_gap@10,diversity@10,recommended_unique_items,n_items
0,Bayram-style ALS paid minutes,61566,0.291833,0.291833,0.029183,0.291833,0.174393,0.194967,0.002895,0.913915,1472,7550


,user_idx,hidden_item_idx,hit,precision@10,recall@10,ndcg@10,diversity@10
0,0,15,1,0.1,1,1.000000,0.936475
1,1,27,0,0.0,0,0.000000,0.934402
2,2,34,0,0.0,0,0.000000,0.922589
3,3,47,0,0.0,0,0.000000,0.550970
4,4,94,1,0.1,1,0.315465,0.970263


In [25]:
bayram_summary = pd.DataFrame([
    {
        "model": "Bayram2021_paid_ALS_minutes_replication",
        "factors": ALS_FACTORS,
        "regularization": ALS_REGULARIZATION,
        "iterations": ALS_ITERATIONS,
        "alpha": ALS_ALPHA,
        "playtime_transform": "alpha * (log(playtime_forever_minutes + 1) + 0.01)",
        "k": K,
        "metric_name": f"Bayram-style Precision@{K} / HitRate@{K}",
        "metric_value": hit_rate_at_10,
        "metric_percent": hit_rate_at_10 * 100,
        "users_evaluated": len(test_items_by_user),
        "train_interactions": user_item_train.nnz,
        "test_interactions": len(test_df),
        "n_users": n_users,
        "n_items": n_items,
    }
])

display(bayram_summary)

bayram_summary.to_csv(
    RESULTS_DIR / "bayram2021_paid_als_minutes_replication_summary.csv",
    index=False,
)

bayram_eval_results.to_csv(
    RESULTS_DIR / "bayram2021_paid_als_minutes_replication_per_user_hits.csv",
    index=False,
)

,model,factors,regularization,iterations,alpha,playtime_transform,k,metric_name,metric_value,metric_percent,users_evaluated,train_interactions,test_interactions,n_users,n_items
0,Bayram2021_paid_ALS_minutes_replication,100,1.0,20,6.0,alpha * (log(playtime_forever_minutes + 1) + 0...,10,Bayram-style Precision@10 / HitRate@10,0.291833,29.183315,61566,3154931,61566,61566,7550


In [26]:
BAYRAM_REPORTED_ALS_PRECISION_10 = 0.3160

bayram_comparison = pd.DataFrame([
    {
        "source": "Bayram reported ALS paid games",
        "metric": "Precision@10 / HitRate@10",
        "value": BAYRAM_REPORTED_ALS_PRECISION_10,
        "percent": BAYRAM_REPORTED_ALS_PRECISION_10 * 100,
        "notes": "reported result",
    },
    {
        "source": "our approximate replication",
        "metric": "Precision@10 / HitRate@10",
        "value": hit_rate_at_10,
        "percent": hit_rate_at_10 * 100,
        "notes": "paid games only, minutes-based log playtime transform, implicit ALS",
    },
])

bayram_comparison["absolute_difference"] = (
    bayram_comparison["value"] - BAYRAM_REPORTED_ALS_PRECISION_10
)

bayram_comparison["relative_difference"] = (
    bayram_comparison["absolute_difference"] / BAYRAM_REPORTED_ALS_PRECISION_10
)

display(bayram_comparison)

bayram_comparison.to_csv(
    RESULTS_DIR / "bayram2021_paid_als_minutes_comparison.csv",
    index=False,
)

,source,metric,value,percent,notes,absolute_difference,relative_difference
0,Bayram reported ALS paid games,Precision@10 / HitRate@10,0.316000,31.600000,reported result,0.000000,0.000000
1,our approximate replication,Precision@10 / HitRate@10,0.291833,29.183315,"paid games only, minutes-based log playtime tr...",-0.024167,-0.076477


In [27]:
def evaluate_extended_popularity_one_hidden_item(
    train_df,
    test_items_by_user,
    n_items,
    k=10,
):
    train_item_counts = train_df["item_idx"].value_counts()
    train_item_distribution = train_item_counts / train_item_counts.sum()

    popular_items = train_item_counts.index.to_numpy()

    train_items_by_user = (
        train_df
        .groupby("user_idx")["item_idx"]
        .apply(set)
        .to_dict()
    )

    recommended_all = []

    hit_values = []
    precision_values = []
    recall_values = []
    ndcg_values = []

    recommended_popularity_values = []

    per_user_rows = []

    for user_idx, hidden_item_idx in test_items_by_user.items():
        already_seen = train_items_by_user.get(user_idx, set())

        recommended_items = [
            item_idx
            for item_idx in popular_items
            if item_idx not in already_seen
        ][:k]

        hit = int(hidden_item_idx in recommended_items)
        precision = hit / k
        recall = hit
        ndcg = ndcg_one_hidden_item(
            recommended_items,
            hidden_item_idx,
            k=k,
        )

        recommended_all.extend(recommended_items)

        item_popularities = [
            train_item_distribution.get(item_idx, 0.0)
            for item_idx in recommended_items
        ]

        recommended_popularity_values.extend(item_popularities)

        hit_values.append(hit)
        precision_values.append(precision)
        recall_values.append(recall)
        ndcg_values.append(ndcg)

        per_user_rows.append({
            "user_idx": user_idx,
            "hidden_item_idx": hidden_item_idx,
            "hit": hit,
            f"precision@{k}": precision,
            f"recall@{k}": recall,
            f"ndcg@{k}": ndcg,
        })

    recommended_unique_items = len(set(recommended_all))
    coverage_at_k = recommended_unique_items / n_items

    gap_recommendations = np.mean(recommended_popularity_values)

    all_item_popularities = np.zeros(n_items)

    for item_idx, popularity in train_item_distribution.items():
        all_item_popularities[item_idx] = popularity

    gap_catalog = np.mean(all_item_popularities)
    delta_gap_at_k = gap_recommendations - gap_catalog

    summary = {
        "model": "Popularity",
        "users_evaluated": len(test_items_by_user),
        f"hit_rate@{k}": np.mean(hit_values),
        f"bayram_precision@{k}": np.mean(hit_values),
        f"precision@{k}": np.mean(precision_values),
        f"recall@{k}": np.mean(recall_values),
        f"ndcg@{k}": np.mean(ndcg_values),
        f"coverage@{k}": coverage_at_k,
        f"delta_gap@{k}": delta_gap_at_k,
        f"diversity@{k}": np.nan,
        "recommended_unique_items": recommended_unique_items,
        "n_items": n_items,
    }

    per_user_df = pd.DataFrame(per_user_rows)

    return summary, per_user_df

In [28]:
popularity_extended_summary, popularity_extended_per_user = evaluate_extended_popularity_one_hidden_item(
    train_df=train_df,
    test_items_by_user=test_items_by_user,
    n_items=n_items,
    k=K,
)

bayram_als_vs_popularity_extended = pd.DataFrame([
    bayram_extended_summary,
    popularity_extended_summary,
])

display(bayram_als_vs_popularity_extended)

bayram_als_vs_popularity_extended.to_csv(
    RESULTS_DIR / "bayram2021_paid_als_minutes_vs_popularity_extended_metrics.csv",
    index=False,
)

popularity_extended_per_user.to_csv(
    RESULTS_DIR / "bayram2021_paid_popularity_extended_per_user_metrics.csv",
    index=False,
)

,model,users_evaluated,hit_rate@10,bayram_precision@10,precision@10,recall@10,ndcg@10,coverage@10,delta_gap@10,diversity@10,recommended_unique_items,n_items
0,Bayram-style ALS paid minutes,61566,0.291833,0.291833,0.029183,0.291833,0.174393,0.194967,0.002895,0.913915,1472,7550
1,Popularity,61566,0.249017,0.249017,0.024902,0.249017,0.164794,0.020795,0.006094,NaN,157,7550


In [29]:
replication_note = f"""
Bayram 2021 paid-games ALS replication note:

This notebook approximately replicates the paid-games ALS setting from Bayram's
Steam recommender project.

The experiment uses only paid games, defined as games with price_numeric_clean > 0
in steam_games_final.csv. Missing prices and non-numeric price values were converted
to 0, therefore such games were excluded from the paid-games subset.

The model parameters are:
- factors = {ALS_FACTORS}
- regularization = {ALS_REGULARIZATION}
- iterations = {ALS_ITERATIONS}
- alpha = {ALS_ALPHA}
- top-k = {K}

The main replication variant uses the following confidence transformation:

    confidence = alpha * (log(playtime_forever + 1) + {PLAYTIME_ADD_CONSTANT})

where playtime_forever is used in the original unit available in users_items.csv,
i.e. minutes.

Although the text of the replicated work describes playtime in hours, the
hours-based variant produced a substantially lower HitRate@{K}. The minutes-based
variant produced a result closer to the reported paid-games ALS result. This suggests
that either the original implementation used the raw playtime_forever scale, or the
unit description in the text is ambiguous.

The evaluation follows a one-hidden-item protocol: one item is hidden for each user,
and the model is evaluated by checking whether this hidden item appears in the top-{K}
recommendations.

Therefore the metric reported by Bayram as Precision@{K} is interpreted here as
HitRate@{K}, not classical Precision@{K}. In this protocol:
- Recall@{K} equals HitRate@{K}, because each user has exactly one hidden test item.
- Classical Precision@{K} equals HitRate@{K} / {K}.
- NDCG@{K} is additionally reported to account for the rank of the hidden item.
"""

print(replication_note)

with open(
    RESULTS_DIR / "bayram2021_paid_als_minutes_replication_note.txt",
    "w",
    encoding="utf-8",
) as f:
    f.write(replication_note)

print("Saved note:", RESULTS_DIR / "bayram2021_paid_als_minutes_replication_note.txt")


Bayram 2021 paid-games ALS replication note:

This notebook approximately replicates the paid-games ALS setting from Bayram's
Steam recommender project.

The experiment uses only paid games, defined as games with price_numeric_clean > 0
in steam_games_final.csv. Missing prices and non-numeric price values were converted
to 0, therefore such games were excluded from the paid-games subset.

The model parameters are:
- factors = 100
- regularization = 1.0
- iterations = 20
- alpha = 6.0
- top-k = 10

The main replication variant uses the following confidence transformation:

    confidence = alpha * (log(playtime_forever + 1) + 0.01)

where playtime_forever is used in the original unit available in users_items.csv,
i.e. minutes.

Although the text of the replicated work describes playtime in hours, the
hours-based variant produced a substantially lower HitRate@10. The minutes-based
variant produced a result closer to the reported paid-games ALS result. This suggests
that either the origi